In [7]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')
import pandas as pd
import numpy as np 
import warnings
warnings.filterwarnings('ignore')
from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
db = DuckDBClient()
s  = S3Client()

In [ ]:
class USDIRNFeatureBuilding:
    def __init__(self, instrument=None):
        self.instrument = instrument
        self._load_data()
    def _load_data(self):
        all_files  = s.list_files(f'processed/1s/{self.instrument}26SEPFUT/')
        all_files += s.list_files(f'processed/1s/{self.instrument}26OCTFUT/')
        all_df = []
        for f in all_files:
            parts = f.split('/')
            sym   = parts[2]
            date  = parts[3].replace('.parquet','')
            try:
                df = db.read_parquet(f'processed/1s/{sym}/{date}.parquet')
                df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
                df = df[(df['ist_sec'] >= 9*3600) &
                        (df['ist_sec'] <= 17*3600)].reset_index(drop=True)
                df['date']   = date
                df['symbol'] = sym
                all_df.append(df)
                print(f'{sym}/{date}: {len(df)} bars')
            except Exception as e:
                print(f'ERROR {sym}/{date}: {e}')
        
        db.close()
        df_raw = pd.concat(all_df, ignore_index=True).reset_index(drop=True)
        print(f'\nTotal: {len(df_raw):,} bars across {df_raw["date"].nunique()} days')
        print(f'Dates: {sorted(df_raw["date"].unique())}')
        print(f'Columns: {len(df_raw.columns)}')
        return df_raw
    def pre_processing_raw(self):
        raw_data = self._load_data()
        df = (raw_data.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True))
        print(df.shape)
        print(df["date"].nunique())
        print(df.head())
        # Verifying duplicate timestamps
        duplicates = df.duplicated(subset=["symbol","date","ts_sec"],keep=False)
        print("Duplicate symbol-date-second rows:", duplicates.sum())
        # Regularization by making original observations
        df["is_observed"] = 1
        df["had_update"] = 1
        # Regularized each trading day at 1-second time clock
        def regularize_day(day):
            day = day.sort_values("ts_sec").copy()
            start = int(day["ts_sec"].min())
            end = int(day["ts_sec"].max())
            full_seconds = pd.DataFrame({"ts_sec": np.arange(start,end + 1)})
            symbol = day["symbol"].iloc[0]
            date = day["date"].iloc[0]
            full_seconds["symbol"] = symbol
            full_seconds["date"] = date
            out = full_seconds.merge(day,on=["symbol","date","ts_sec"], how="left")
            return out
        df_1s = (df.groupby(["symbol", "date"],group_keys=False)
            .apply(regularize_day)
            .reset_index(drop=True))
        #Fixing is_observed and had update 
        df_1s["is_observed"] = (df_1s["is_observed"].fillna(0).astype(int))
        df_1s["had_update"] = (df_1s["had_update"].fillna(0).astype(int))
        state_cols = [
                "oi",
            
                "bid_p1", "bid_q1",
                "bid_p2", "bid_q2",
                "bid_p3", "bid_q3",
                "bid_p4", "bid_q4",
                "bid_p5", "bid_q5",
            
                "ask_p1", "ask_q1",
                "ask_p2", "ask_q2",
                "ask_p3", "ask_q3",
                "ask_p4", "ask_q4",
                "ask_p5", "ask_q5",
            ]
        state_cols = [c for c in state_cols if c in df_1s.columns]
        df_1s[state_cols] = (df_1s.groupby(["symbol", "date"])[state_cols].ffill())
        flow_cols = ["volume_delta","tick_count"]
        for col in flow_cols:
            if col in df_1s.columns:
                df_1s[col] = (df_1s[col].fillna(0))
        df_1s["mid"] = (df_1s["bid_p1"]+df_1s["ask_p1"]) / 2
        df_1s["spread"] = (df_1s["ask_p1"]-df_1s["bid_p1"])
        eps = 1e-12
        df_1s["spread_bps_calc"] = (1e4*df_1s["spread"]/(df_1s["mid"] + eps))
        df_1s["log_mid"] = np.log(df_1s["mid"])
        df_1s["ret_1s"] = (df_1s["log_mid"]-df_1s.groupby(["symbol", "date"])["log_mid"].shift(1))
        bid_cols = [f"bid_q{i}" for i in range(1, 6)]
        ask_cols = [f"ask_q{i}" for i in range(1, 6)]
        df_1s["bid_depth_l5"] = (df_1s[bid_cols].sum(axis=1))
        df_1s["ask_depth_l5"] = (df_1s[ask_cols].sum(axis=1))
        df_1s["total_depth_l5"] = (df_1s["bid_depth_l5"]+df_1s["ask_depth_l5"])
        session_summary = (df_1s.groupby(["symbol", "date"]).agg(
            first_sec=("ts_sec", "min"),
            last_sec=("ts_sec", "max"),
            rows=("ts_sec", "size"),
            observed_rows=("is_observed", "sum")))
        session_summary["observation_rate"] = (session_summary["observed_rows"]/session_summary["rows"])
        print(session_summary)
        return df_1s
    def _feature_calculation(self):
        df_1s = self.pre_processing_raw()
        eps = 1e-12
        df_1s["mid"] = (df_1s["bid_p1"]+ df_1s["ask_p1"]) / 2.0
        df_1s["log_mid"] = np.log(df_1s["mid"])
        g = df_1s.groupby(["symbol", "date"],group_keys=False)
        df_1s["ret_1s"] = (df_1s["log_mid"]- g["log_mid"].shift(1))
        